# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [2]:
import os
os.environ["USER_AGENT"] = "my-langchain-app"

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("../02_activities/documents/managing_oneself.pdf")
docs = loader.load()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(docs)

print(len(chunks))
print(chunks[0].page_content[:300])

68
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materi


In [5]:
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.embeddings.create(
    model = 'text-embedding-3-small',
    input = chunks[0].page_content
    
)
vector = response.data[0].embedding

print(len(vector))

1536


In [7]:
vectors = []
for chunk in chunks:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk.page_content
    )
    vectors.append(response.data[0].embedding)

print(len(vectors))

68


In [8]:
import faiss
import numpy as np
vectors_np = np.array(vectors).astype("float32")

dimension = vectors_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(vectors_np)

print(index.ntotal)

68


In [9]:
query = "What does Drucker say about strengths?"

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_vector = np.array([response.data[0].embedding]).astype("float32")

# Search top 3
D, I = index.search(query_vector, k=3)

print(I)


[[ 8 65 66]]


In [10]:
for i in I[0]:
    print("CHUNK:", i)
    print(chunks[i].page_content[:1000])
    print("-" * 50)

CHUNK: 8
manage ourselves. We will have to learn to de-
velop ourselves. We will have to place our-
selves where we can make the greatest contri-
bution. And we will have to stay mentally alert
and engaged during a 50-year working life,
which means knowing how and when to
change the work we do.
 
What Are My Strengths?
 
Most people think they know what they are
good at. They are usually wrong. More often,
people know what they are not good at—and
even then more people are wrong than right.
And yet, a person can perform only from
strength. One cannot build performance on
weaknesses, let alone on something one can-
not do at all.
Throughout history, people had little
need to know their strengths. A person was
This document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
--------------------------------------------------

In [11]:
context = chunks[8].page_content

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer only from the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: What does Drucker say about managing oneself?"}
    ]
)

print(response.choices[0].message.content)

Drucker emphasizes the importance of self-management in one's career. He suggests that individuals must learn to develop themselves, position themselves to make significant contributions, and remain mentally alert and engaged throughout a lengthy working life. This involves knowing when and how to adapt their work to stay effective.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [12]:
import deepeval
print("ok")

ok


In [16]:
import os

os.environ["OPENAI_API_KEY"] = "any value"
os.environ["OPENAI_BASE_URL"] = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"

In [17]:
summary = response.choices[0].message.content
original_text = chunks[8].page_content

In [25]:
from deepeval.models import DeepEvalBaseLLM

class CustomModel(DeepEvalBaseLLM):
    def load_model(self):
        return client
    
    def generate(self, prompt: str) -> str:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "custom-gateway-model"

In [26]:
from deepeval.metrics import SummarizationMetric
custom_model = CustomModel()

metric = SummarizationMetric(
    model=custom_model,
    assessment_questions=[
        "Does the summary capture the main idea?",
        "Does the summary mention self-management?",
        "Does the summary mention contribution?",
        "Does the summary mention adapting over time?",
        "Is the summary factually consistent?"
    ]
)

In [ ]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=original_text,
    actual_output=summary
)

metric.measure(test_case)

print(metric.score)
print(metric.reason)

Output()

0.0
The score is 0.00 because the summary includes numerous pieces of extra information that are not present in the original text, leading to a significant lack of alignment with the source material.


In [29]:
original_text = (
    chunks[8].page_content +
    chunks[65].page_content +
    chunks[66].page_content
)

test_case = LLMTestCase(
    input=original_text,
    actual_output=summary
)

metric.measure(test_case)
print(metric.score)
print(metric.reason)

Output()

0.0
The score is 0.00 because the summary introduces multiple pieces of extra information that are not present in the original text, which detracts from its fidelity to the source material.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [30]:
improve_prompt = f"""
Original Context:
{original_text}

Current Summary:
{summary}

Evaluation Feedback:
{metric.reason}

Rewrite the summary so it is:
1. More faithful to the context
2. Clear and concise
3. Factually accurate
4. Better aligned with the source text only

Return only the improved summary.
"""

In [31]:
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": improve_prompt}
    ]
)

new_summary = response2.choices[0].message.content

print(new_summary)

Drucker highlights the necessity of understanding one's strengths to enhance performance and contributions in the workplace. He points out that individuals often misjudge their abilities and emphasizes that effective performance can only be built on strengths, not weaknesses. In modern organizations, the focus has shifted from subject matter expertise to personal abilities, such as empathy and resilience. Therefore, managers must recognize and combine the diverse skills of individuals to achieve the best outcomes.


In [32]:
test_case2 = LLMTestCase(
    input=original_text,
    actual_output=new_summary
)

metric.measure(test_case2)

print(metric.score)
print(metric.reason)

Output()

0.6
The score is 0.60 because the summary includes misleading information that contradicts the original text, specifically regarding the focus shift in measurement of competence. Additionally, it fails to address key aspects such as self-management and adaptation, which the original text references, indicating that it lacks completeness.


The second version was better. The score improved from 0.0 to 0.6, which shows that using evaluation feedback helped improve the summary. It became more aligned with the original text and reduced some of the earlier mistakes.

However, it is still was not perfect. Some important ideas were still missing, and a few parts were slightly misleading. So while this feedback loop definitely helps, it is not enough on its own.

In a real system, I would also add better document retrieval, clearer prompts, multiple evaluation checks, and human review when accuracy really matters.

Overall, this exercise shows that AI outputs can improve through self-correction, but quality control should use more than just one method.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
